# Notebook 02 — Baseline Evaluation on Real Access Logs

**Fixes vs original:**
- **E2E latency corrected**: measured on a *single request* (not 698k entries)
- **min_length filter**: before/after counts verified — zero attacks removed
- **Benign pool saves query values** (not raw URLs) so NB03 can use them directly
- Parsing documented with example log lines → decoded output
- FP/10k and alerts/day at default threshold and T_high
- 3-tier decision stub (full implementation in NB05)


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import joblib, re, os, time, json, urllib.parse, warnings
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['results/metrics', 'results/figures',
          'results/attack_logs', 'results/benign_pool']:
    os.makedirs(d, exist_ok=True)

SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

def extract_query_values(url):
    """Return joined query param VALUES, or None if no query string."""
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
    values = [urllib.parse.unquote(v).strip()
              for vlist in params.values() for v in vlist
              if urllib.parse.unquote(v).strip()]
    return ' '.join(values) if values else None

METHOD_RE = re.compile(r'"(GET|POST|HEAD|PUT|DELETE|OPTIONS|PATCH|TRACE|CONNECT)\s+([^"]+)\s+HTTP')

print('Setup complete.')


## 2. Log Format Documentation

```
<line_no>,<label>,<ip> - - [timestamp] "METHOD URL HTTP/x" status bytes "ref" "ua"
```
Label: 0 = benign, 1 = SQLi attack.  Three examples parsed below.


In [ ]:
EXAMPLES = [
    # Benign GET request — synthetic/anonymized example
    ('1,0,192.0.2.10 - - [30/Dec/2024:03:37:01 +0300]'
     ' "GET /index.php/example/location/electronics/ad/19519415 HTTP/1.1"'
     ' 200 214097 "-" "Googlebot/2.1"'),

    # Benign HEAD request with query parameter — synthetic/anonymized example
    ('3,0,198.51.100.20 - - [30/Dec/2024:03:37:02 +0300]'
     ' "HEAD /example/all/items/ad/18575020?page=106 HTTP/1.1"'
     ' 200 0 "-" "Chrome/71.0"'),

    # Synthetic SQLi request
    ('702381,1,203.0.113.30 - - [31/Dec/2024:23:59:58 +0300]'
     ' "GET /products?id=1\'%20UNION%20SELECT%20username,password%20FROM%20users-- HTTP/1.1"'
     ' 200 512 "-" "sqlmap/1.0"'),
]

print('Parsing examples:')
print('=' * 60)
for raw in EXAMPLES:
    parts     = raw.strip().split(',', 2)
    line_no, label, log_entry = parts
    m         = METHOD_RE.search(log_entry)
    url_raw   = m.group(2) if m else '(not found)'
    url_dec   = urllib.parse.unquote(url_raw)
    url_dec   = re.sub(r'utm_[a-z]+=[^&]*', '', url_dec, flags=re.IGNORECASE).strip()
    qv        = extract_query_values(url_dec)
    print(f'  Line    : {line_no}  Label: {label}')
    print(f'  Raw URL : {url_raw[:80]}')
    print(f'  Decoded : {url_dec[:80]}')
    print(f'  QValues : {repr(qv)}')
    print()


## 3. Load & Parse Log

In [ ]:
records = []
with open('../logs/labeled_access.log', 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split(',', 2)
        if len(parts) == 3:
            records.append(parts)

df = pd.DataFrame(records, columns=['Line', 'True_Label', 'Log_Entry'])
df['Line']       = df['Line'].astype(int)
df['True_Label'] = df['True_Label'].astype(int)

total   = len(df)
attacks = int(df['True_Label'].sum())
benign  = total - attacks
print(f'Total   : {total:,}')
print(f'Benign  : {benign:,}  ({benign/total*100:.4f}%)')
print(f'Attack  : {attacks:,}  ({attacks/total*100:.4f}%)')
print(f'Rate    : 1 in every {total // attacks:,} requests')

ts_re = re.compile(r'\[(\d{2}/\w+/\d{4})')
dates = sorted(set(
    ts_re.search(str(e)).group(1)
    for e in df['Log_Entry'].head(5000)
    if ts_re.search(str(e))
))
print(f'Log dates (sample): {dates}')


## 4. URL Extraction & min_length Filter

`min_length=10` filters on the full URL path.
Paths shorter than 10 chars are bare roots (`/`, `/en`) — they cannot carry SQLi.
We verify that zero attack entries are removed.


In [ ]:
df['Query'] = (
    df['Log_Entry']
    .apply(lambda x: (METHOD_RE.search(str(x)) or [None, None, None])[2] or '')
    .apply(lambda x: re.sub(r'utm_[a-z]+=[^&]*', '',
                             urllib.parse.unquote(str(x)),
                             flags=re.IGNORECASE).strip())
)

min_length     = 10
before_total   = len(df)
before_attacks = int(df['True_Label'].sum())

df = df[df['Query'].str.len() >= min_length].reset_index(drop=True)
after_total    = len(df)
after_attacks  = int(df['True_Label'].sum())

print(f'BEFORE : {before_total:,} total | {before_attacks} attacks')
print(f'AFTER  : {after_total:,} total  | {after_attacks} attacks')
print(f'Removed: {before_total - after_total:,} entries | {before_attacks - after_attacks} attacks')
print()
if before_attacks == after_attacks:
    print('✅ min_length=10 removes ZERO attack entries.')
    print('   Removed entries are bare root/asset paths with no SQLi capacity.')
else:
    print('⚠️  WARNING: attacks removed — investigate!')


## 5. Vectorize with NB01 Vocabulary

In [ ]:
vectorizer  = joblib.load('results/models/vectorizer_with_symbols.pkl')
queries     = df['Query'].tolist()
y_true      = df['True_Label'].values
BENIGN_SIZE = int((y_true == 0).sum())
TOTAL_ATK   = int(y_true.sum())

X_text = vectorizer.transform(queries)
X_sym  = build_symbol_matrix(queries)
X_real = hstack([X_text, X_sym])

print(f'Vocabulary : {len(vectorizer.vocabulary_):,}  Total features: {X_real.shape[1]:,}')
print(f'Samples    : {X_real.shape[0]:,}')


## 6. Run All Models

**E2E latency fix:** timing is measured on **one request** (vectorize + predict),
not on the full 698k-entry dataset.


In [ ]:
model_files = {
    'Logistic Regression': 'results/models/logistic_regression_model_with_symbols.pkl',
    'SGD (log loss)':      'results/models/sgd_log_loss_model_with_symbols.pkl',
    'LinearSVC':           'results/models/linearsvc_model_with_symbols.pkl',
    'Decision Tree':       'results/models/decision_tree_model_with_symbols.pkl',
    'Naive Bayes':         'results/models/naive_bayes_model_with_symbols.pkl',
    'Random Forest':       'results/models/random_forest_model_with_symbols.pkl',
}

sample_q = [queries[0]]      # single request for E2E timing
results, all_preds, all_probs = [], {}, {}

for name, path in model_files.items():
    model  = joblib.load(path)
    y_pred = model.predict(X_real)
    y_prob = model.predict_proba(X_real)[:, 1]

    # Single-request E2E: vectorize + predict
    t0 = time.perf_counter()
    Xs = hstack([vectorizer.transform(sample_q), build_symbol_matrix(sample_q)])
    model.predict(Xs)
    e2e_ms = (time.perf_counter() - t0) * 1000

    tp    = int(((y_pred==1) & (y_true==1)).sum())
    fp    = int(((y_pred==1) & (y_true==0)).sum())
    fn    = int(((y_pred==0) & (y_true==1)).sum())
    prec  = precision_score(y_true, y_pred, zero_division=0)
    rec   = recall_score(y_true, y_pred, zero_division=0)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    fp10k = fp / BENIGN_SIZE * 10000

    results.append({
        'Model':       name,
        'TP':          tp,   'FP': fp,   'FN': fn,
        'Recall':      round(rec,  4),
        'Precision':   round(prec, 4),
        'F1':          round(f1,   4),
        'FP_per_10k':  round(fp10k, 1),
        'E2E_ms':      round(e2e_ms, 3),
    })
    all_preds[name] = y_pred
    all_probs[name] = y_prob
    print(f'{name}: TP={tp}  FP={fp:,}  Rec={rec:.4f}  FP/10k={fp10k:.1f}  E2E={e2e_ms:.3f}ms')

results_df = pd.DataFrame(results)
results_df.to_csv('results/metrics/02_model_results_real_logs.csv', index=False)
print()
print('=== SUMMARY (threshold 0.5) ===')
print(results_df[['Model', 'TP', 'FP', 'Recall', 'FP_per_10k', 'E2E_ms']].to_string(index=False))


## 7. Threshold Analysis — RF at T_high and T_low

In [ ]:
th       = json.load(open('results/models/01_thresholds.json'))
rf_prob  = all_probs['Random Forest']

print(f'T_high={th["t_high"]}  T_low={th["t_low"]}  '
      f'(T_high > T_low: {th["t_high"] > th["t_low"]}  ✅)')
print()

for lbl, tv in [('T_high', th['t_high']), ('T_low', th['t_low'])]:
    yp    = (rf_prob >= tv).astype(int)
    tp_   = int(((yp==1) & (y_true==1)).sum())
    fp_   = int(((yp==1) & (y_true==0)).sum())
    prec_ = tp_ / (tp_ + fp_) if tp_ + fp_ > 0 else 0
    rec_  = tp_ / TOTAL_ATK
    print(f'RF at {lbl}={tv}: TP={tp_}  FP={fp_:,}  '
          f'Prec={prec_:.4f}  Rec={rec_:.4f}  FP/10k={fp_/BENIGN_SIZE*10000:.2f}')

print()
print('=== 3-TIER STUB (full implementation in NB05) ===')
tv_h = th['t_high'];  tv_l = th['t_low']
n_atk  = int((rf_prob >= tv_h).sum())
n_susp = int(((rf_prob >= tv_l) & (rf_prob < tv_h)).sum())
n_ben  = int((rf_prob < tv_l).sum())
tp_atk  = int(((rf_prob >= tv_h) & (y_true==1)).sum())
tp_susp = int(((rf_prob >= tv_l) & (rf_prob < tv_h) & (y_true==1)).sum())
print(f'  ATTACK     (>={tv_h})           : {n_atk:,}  TP={tp_atk}')
print(f'  SUSPICIOUS ({tv_l} <= s < {tv_h}): {n_susp:,}  TP={tp_susp}')
print(f'  BENIGN     (<{tv_l})            : {n_ben:,}')


## 8. High-Confidence Benign Pool for NB03

**Critical fix:** we now save the **extracted query values**, not raw URLs.
The previous version saved full URL paths; NB03's `extract_query_values()`
returned `None` for all of them (path-only URLs have no `?param=value`),
leaving zero pseudo-labeled negatives.


In [ ]:
BENIGN_THR  = 0.05
rf_prob     = all_probs['Random Forest']
hc_mask     = (rf_prob < BENIGN_THR) & (y_true == 0)
hc_df       = df[hc_mask].copy()
hc_df['RF_score']    = rf_prob[hc_mask]

# Extract query values NOW — save only rows that have them
hc_df['QueryValues'] = hc_df['Query'].apply(extract_query_values)
hc_qv = hc_df[hc_df['QueryValues'].notna()].drop_duplicates('QueryValues')

print(f'Entries with RF < {BENIGN_THR}       : {len(hc_df):,}')
print(f'  → with query values (usable) : {len(hc_qv):,}')
print(f'  → true attacks in pool       : {int((hc_qv["True_Label"]==1).sum())}  (must be 0)')
print()
print('Sample query values:')
for _, row in hc_qv.sample(min(10, len(hc_qv)), random_state=42).iterrows():
    print(f'  score={row["RF_score"]:.3f}  {repr(row["QueryValues"][:60])}')

hc_qv[['RF_score', 'QueryValues']].to_csv(
    'results/benign_pool/02_hc_benign_qvalues.csv', index=False)
print()
print('Saved: results/benign_pool/02_hc_benign_qvalues.csv')
print('→ NB03 loads QueryValues column directly as pseudo-labeled negatives')


## 9. Missed Attacks & Figures

In [ ]:
rf_preds = all_preds['Random Forest']
missed   = df[(y_true == 1) & (rf_preds == 0)]
print(f'=== RF Missed Attacks (FN={len(missed)}) ===')
for _, row in missed.iterrows():
    sc = all_probs['Random Forest'][row.name]
    print(f'  [Line {row["Line"]}] score={sc:.3f}  {row["Query"][:90]}')

# FP/10k bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(results_df['Model'], results_df['FP_per_10k'], color='tomato')
ax.set_ylabel('False Positives per 10,000 Requests')
ax.set_title('Baseline FP Rate — NB01 Models on Real Logs')
ax.grid(axis='y', alpha=0.4); ax.tick_params(axis='x', rotation=20)
for b, v in zip(bars, results_df['FP_per_10k']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f'{v:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('results/figures/02_fp_per_10k.png', dpi=150)
plt.close()

# Confusion matrices
fig, axes = plt.subplots(1, len(all_preds), figsize=(5*len(all_preds), 4))
for ax, (name, yp) in zip(axes, all_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_true, yp),
                           display_labels=['legit', 'attack']).plot(
        cmap=plt.cm.Blues, ax=ax, colorbar=False)
    ax.set_title(name, fontsize=8)
plt.suptitle('Confusion Matrices — Real Logs (Baseline)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/02_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()

print('Saved: 02_fp_per_10k.png  02_confusion_matrices.png')


## 10. Summary

In [ ]:
th = json.load(open('results/models/01_thresholds.json'))
rf_p  = all_probs['Random Forest']
fp_h  = int(((rf_p >= th['t_high']) & (y_true == 0)).sum())
tp_h  = int(((rf_p >= th['t_high']) & (y_true == 1)).sum())

print('=' * 65)
print('NOTEBOOK 02 — COMPLETE (Baseline)')
print('=' * 65)
print(f'Log: {len(df):,} entries | {TOTAL_ATK} attacks | {BENIGN_SIZE:,} benign')
print()
print('RESULTS (threshold 0.5):')
print(results_df[['Model', 'TP', 'FP', 'Recall', 'FP_per_10k', 'E2E_ms']].to_string(index=False))
print()
print(f'RF at T_high={th["t_high"]}: TP={tp_h}  FP={fp_h:,}  FP/10k={fp_h/BENIGN_SIZE*10000:.2f}')
print()
print(f'Benign pool (with query values): {len(hc_qv):,} → NB03')
print()
print('ROOT CAUSE: NB01 trained on bare SQL, NB02 infers on full URL paths.')
print('FIX (NB03): align training and inference on extracted query values.')
print()
print('NEXT: Notebook 03 — Retrain with bootstrapped negatives (Way 3)')
